# INITIAL LOAD mobile_measurements_autoloader

In [0]:
from pyspark.sql.functions import current_timestamp

checkpoint_path = "abfss://data-lake@storagetelcopoc2026.dfs.core.windows.net/01-bronze/_checkpoints/mobile_measurements_autoloader"

(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path + "/schema")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .load("/Volumes/telco_prod/00-raw(ext)/volume_raw")
    .select("*", current_timestamp().alias("_ingested_at"))
    .writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)    
    .format("delta")
    #.option("path", "abfss://data-lake@storagetelcopoc2026.dfs.core.windows.net/01-bronze/mobile_measurements_autoloader")
    .toTable("telco_prod.`01-bronze`.mobile_measurements_autoloader")
    #.start()
)

## Check

In [0]:
df = spark.table("telco_prod.`01-bronze`.mobile_measurements_autoloader")
df.count()
#df.display()

In [0]:
df.groupBy("processing_date").count().orderBy("processing_date").display()

In [0]:
%sql
select * from `telco_prod`.`01-bronze`.`mobile_measurements_autoloader` limit 100;

select processing_date, is_outlier, count(*)
from telco_prod.`01-bronze`.mobile_measurements_autoloader
group by processing_date, is_outlier
order by 1, 2;

## Check parquet files directly on storage

In [0]:
path = "abfss://data-lake@storagetelcopoc2026.dfs.core.windows.net/01-bronze/mobile_measurements_autoloader"
direct_df = spark.read.format("delta").load(path)
print(f"Direktno sa Azure-a: {direct_df.count()}")

Recreate table if UNITY CATALOG dosn't see new data

In [0]:
%sql
DROP TABLE IF EXISTS telco_prod.`01-bronze`.mobile_measurements_autoloader;

-- Table creation using the same location
CREATE TABLE telco_prod.`01-bronze`.mobile_measurements_autoloader
USING DELTA
LOCATION 'abfss://data-lake@storagetelcopoc2026.dfs.core.windows.net/01-bronze/mobile_measurements_autoloader';